[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module2/03-context-managers.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module2/03-context-managers.ipynb)

# Context Managers
**Module 2 — Intermediate Python | Estimated time: 30 minutes**

## Learning Objectives
- Understand the **`with` statement** and why it matters for resource safety
- Implement the **`__enter__` / `__exit__` protocol** in a custom class
- Handle exceptions inside `__exit__` to suppress or re-raise them
- Use **`@contextlib.contextmanager`** to write generator-based context managers
- Apply `contextlib.suppress` and `contextlib.redirect_stdout`
- Build practical context managers: database connection mock, temporary directory, in-memory buffer

In [ ]:
import time
import sys
import io
import os
import tempfile
import contextlib
from typing import Optional

print('Setup complete.')

## 1. Why Context Managers?

Resources like files, locks, and network connections must be **released** even when errors occur.  
The `with` statement guarantees cleanup code runs regardless of how the block exits.

In [ ]:
import tempfile, os

# Without a context manager — easy to forget close() on error
f = open('/tmp/demo.txt', 'w')
try:
    f.write('hello')
finally:
    f.close()          # must be in finally or a bug leaves file open

# With a context manager — clean and safe
with open('/tmp/demo.txt', 'w') as f:
    f.write('hello')   # f.close() is called automatically no matter what

# The with block called __enter__ (returned f) then __exit__ (closed the file)
print('File closed?', f.closed)

# Even on error:
try:
    with open('/tmp/demo.txt', 'w') as f:
        f.write('oops')
        raise RuntimeError('something went wrong')
except RuntimeError:
    pass
print('File closed after error?', f.closed)

## 2. Custom Context Manager — `__enter__` and `__exit__`

Any class that implements both `__enter__` and `__exit__` can be used with `with`.  
`__exit__` receives exception info (type, value, traceback) — if it returns `True` the exception is suppressed.

In [ ]:
import time

class Timer:
    """Context manager that measures elapsed time for a block of code."""

    def __init__(self, name: str = 'block'):
        self.name = name
        self.elapsed: float = 0.0

    def __enter__(self) -> 'Timer':
        """Called on entry to the with block. Return value is bound to `as` target."""
        self._start = time.perf_counter()
        print(f'[Timer] Starting "{self.name}"...')
        return self                          # the object bound to `as t`

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        """Called on exit. Return True to suppress exceptions, False/None to re-raise."""
        self.elapsed = time.perf_counter() - self._start
        status = 'ERROR' if exc_type else 'OK'
        print(f'[Timer] "{self.name}" finished in {self.elapsed:.4f}s [{status}]')
        return False    # never suppress exceptions — let them propagate


# Normal exit
with Timer('matrix multiply') as t:
    total = sum(i * j for i in range(1000) for j in range(100))
print(f'Elapsed stored: {t.elapsed:.4f}s')

# Exit due to exception — __exit__ still runs
try:
    with Timer('failing block'):
        time.sleep(0.05)
        raise ValueError('intentional error')
except ValueError:
    print('Caught the ValueError outside the with block')

## 3. Suppressing Exceptions in `__exit__`

Return `True` from `__exit__` to silence a specific exception type.

In [ ]:
class SuppressErrors:
    """Context manager that silently swallows specified exception types."""

    def __init__(self, *exception_types):
        self.exception_types = exception_types
        self.suppressed: Optional[Exception] = None

    def __enter__(self) -> 'SuppressErrors':
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        if exc_type and issubclass(exc_type, self.exception_types):
            self.suppressed = exc_val
            print(f'  [SuppressErrors] Suppressed: {exc_type.__name__}: {exc_val}')
            return True   # suppress — execution continues after the with block
        return False      # re-raise anything else


with SuppressErrors(FileNotFoundError, KeyError) as ctx:
    open('/nonexistent/path/file.txt')   # raises FileNotFoundError

print('Execution continues after suppressed exception.')
print('Suppressed:', ctx.suppressed)

# contextlib.suppress is the stdlib equivalent — simpler to use
with contextlib.suppress(ZeroDivisionError):
    result = 1 / 0          # silently ignored
print('contextlib.suppress worked.')

## 4. Generator-Based Context Managers — `@contextlib.contextmanager`

Writing a class is overkill for simple cases. The `@contextmanager` decorator turns a generator function into a context manager:  
- Code **before** `yield` → `__enter__`  
- The **`yield`** value → the `as` target  
- Code **after** `yield` → `__exit__`

In [ ]:
import contextlib
import time

@contextlib.contextmanager
def timer(name: str = 'block'):
    """Generator-based equivalent of the Timer class above."""
    start = time.perf_counter()
    print(f'[timer] Starting "{name}"...')
    try:
        yield                            # hand control to the with-block
    finally:
        elapsed = time.perf_counter() - start
        print(f'[timer] "{name}" done in {elapsed:.4f}s')


with timer('list comprehension'):
    data = [x ** 2 for x in range(500_000)]


@contextlib.contextmanager
def managed_resource(name: str):
    """Simulate acquiring and releasing a generic resource."""
    print(f'  Acquiring {name}...')
    resource = {'name': name, 'active': True}
    try:
        yield resource
    except Exception as exc:
        print(f'  Error while using {name}: {exc}')
        raise
    finally:
        resource['active'] = False
        print(f'  Released {name}.')

with managed_resource('GPU memory') as res:
    print(f'  Using resource: {res}')

## 5. `contextlib.redirect_stdout` — Capturing Output

Useful for testing functions that print to stdout, or for capturing CLI output.

In [ ]:
import io
import contextlib

def noisy_function():
    print('Step 1: loading data...')
    print('Step 2: processing...')
    print('Step 3: done. Result = 42')
    return 42


# Capture everything noisy_function prints
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    result = noisy_function()

captured = buffer.getvalue()
print('Return value:', result)
print('Captured output:')
for line in captured.splitlines():
    print(' |', line)

# Redirecting to /dev/null (suppress all output)
with contextlib.redirect_stdout(io.StringIO()):
    print('This will not appear anywhere.')
print('Output suppressed successfully.')

## 6. Practical — Database Connection Mock

A realistic pattern: wrapping a (simulated) database connection so it commits on success and rolls back on error.

In [ ]:
import contextlib

class FakeDB:
    """Simulated database with commit/rollback semantics."""

    def __init__(self, dsn: str):
        self.dsn = dsn
        self.log = []

    def execute(self, sql: str):
        self.log.append(('execute', sql))
        print(f'  DB> {sql}')

    def commit(self):
        self.log.append(('commit',))
        print('  DB> COMMIT')

    def rollback(self):
        self.log.append(('rollback',))
        print('  DB> ROLLBACK')

    def close(self):
        print(f'  DB> Connection to {self.dsn} closed.')


@contextlib.contextmanager
def database(dsn: str):
    """Yield a DB connection; commit on success, rollback on error, always close."""
    db = FakeDB(dsn)
    print(f'Connecting to {dsn}...')
    try:
        yield db
        db.commit()
    except Exception as exc:
        db.rollback()
        raise
    finally:
        db.close()


# Successful transaction
print('=== Successful transaction ===')
with database('postgresql://localhost/mydb') as db:
    db.execute('INSERT INTO users VALUES (1, "Alice")')
    db.execute('UPDATE accounts SET balance = balance - 100 WHERE id = 1')

# Failed transaction
print('\n=== Failed transaction ===')
try:
    with database('postgresql://localhost/mydb') as db:
        db.execute('INSERT INTO orders VALUES (99, 1, 250.00)')
        raise ValueError('Payment gateway timed out')
except ValueError as e:
    print(f'Transaction aborted: {e}')

## 7. Multiple Context Managers and `contextlib.ExitStack`

You can use several `with` targets on one line. `ExitStack` is useful when the number of managers is dynamic.

In [ ]:
import contextlib
import tempfile
import os

# Multiple managers on one line
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as src, \
     tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as dst:
    src.write('source content')
    dst.write('destination content')
    print(f'src: {src.name}')
    print(f'dst: {dst.name}')
os.unlink(src.name); os.unlink(dst.name)


# ExitStack — dynamic number of context managers
def open_many(paths: list[str]):
    """Open all files, return handles; close all on exit even if some fail."""
    with contextlib.ExitStack() as stack:
        handles = [stack.enter_context(open(p, 'w')) for p in paths]
        for i, fh in enumerate(handles):
            fh.write(f'file {i}')
        print(f'Opened {len(handles)} files inside ExitStack')
    print('All files closed by ExitStack')

tmp_paths = [f'/tmp/pypath_demo_{i}.txt' for i in range(4)]
open_many(tmp_paths)
for p in tmp_paths:
    with contextlib.suppress(FileNotFoundError):
        os.unlink(p)

## Practice Exercises

**Exercise 1 — `@contextmanager` Temporary Directory**  
Write a `@contextlib.contextmanager` called `temp_working_dir()` that creates a temporary directory, changes `os.getcwd()` into it for the duration of the block, then changes back and removes the directory when the block exits (even on error).

**Exercise 2 — Retry Context Manager**  
Write a class-based context manager `Retry(max_attempts=3)` that re-enters the body up to `max_attempts` times if a `RuntimeError` is raised. (Hint: this is non-trivial — look up `contextlib.contextmanager` with a loop.) Alternatively, implement it so `__exit__` tracks attempt count.

**Exercise 3 — Capturing and Asserting Logs**  
Write a `@contextmanager` called `capture_logs(logger_name)` that temporarily adds a `logging.Handler` to the named logger, collects all `LogRecord` objects, and yields the list. After the block the handler is removed. Use it to assert that a function emits a specific log message.